<a href="https://colab.research.google.com/github/vikrant48/AI-ML-python-code/blob/main/Crewai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install crewai crewai-tools
!pip install langchain langchain-community langchain-huggingface
!pip install chromadb sentence-transformers pypdf rank_bm25
!pip install langchain-groq

In [ ]:
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY").strip()
os.environ["LANGCHAIN_API_KEY"] = userdata.get("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"]= userdata.get("LANGCHAIN_TRACING_V2")
os.environ["ANONYMIZED_TELEMETRY"] = "False"

In [ ]:
# create chunks

from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=100
)

In [ ]:
# for embedding
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en"   # better for retrieval than MiniLM
)

CrewAI internally uses:

LiteLLM

LiteLLM expects model names like:

groq/llama-3.1-8b-instant
openai/gpt-4o-mini
gemini/gemini-1.5-pro

NOT LangChain objects.

In [ ]:
!pip install -q "crewai[litellm]"
!pip install litellm

In [ ]:
#for base LLM
llm = "groq/llama-3.1-8b-instant"
print(llm)

In [ ]:
import re

def extract_name(text):
    lines = text.split("\n")

    for line in lines[:5]:  # check first 5 lines
        line = line.strip()

        # simple rule: name = line with only alphabets and spaces
        if re.match(r"^[A-Za-z ]{3,}$", line):
            return line

    return "Unknown"

In [ ]:
# here for multiple resume (load pdf and chunking )
from langchain_community.document_loaders import PyPDFLoader

resume_files = [
    "resume1.pdf",
    "resume2.pdf",
    "resume3.pdf"
]
# for store chunking
chunks = []

for file in resume_files:
    loader = PyPDFLoader(file)
    docs = loader.load()

    if not docs:
        print(f"Skipping empty file: {file}")
        continue

    first_page_text = docs[0].page_content
    name = extract_name(first_page_text)
    print("Extracted Name:", name)

    for doc in docs:
        doc.metadata["name"] = name
        doc.metadata["source"] = file   # useful later

    file_chunk = splitter.split_documents(docs)

    chunks.extend(file_chunk)

In [ ]:
#create and store
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

print("Vector DB created")

In [ ]:
#Create BM25
from rank_bm25 import BM25Okapi

corpus_texts = [chunk.page_content for chunk in chunks]
corpus_tokens = [text.lower().split() for text in corpus_texts]

bm25 = BM25Okapi(corpus_tokens)

print("BM25 ready")

In [ ]:
#reranker
from sentence_transformers import CrossEncoder
reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)


In [ ]:
# hybrid search
import numpy as np
def get_dense_rankings(query, k=10):

    dense_docs = vectorstore.similarity_search(query, k=k)

    ranked_indices = []

    for doc in dense_docs:

        for i, chunk in enumerate(chunks):

            if chunk.page_content == doc.page_content:
                ranked_indices.append(i)
                break

    return ranked_indices


def get_bm25_rankings(query, k=10):

    scores = bm25.get_scores(query.lower().split())

    ranked_indices = np.argsort(scores)[::-1][:k]

    return ranked_indices.tolist()


def reciprocal_rank_fusion(rank_lists, k=60):

    scores = {}

    for rank_list in rank_lists:

        for rank, idx in enumerate(rank_list):

            scores[idx] = scores.get(idx, 0) + 1 / (k + rank + 1)

    return sorted(
        scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

In [ ]:
def hybrid_search(query, k=10):

    dense_rank = get_dense_rankings(query, k=10)

    bm25_rank = get_bm25_rankings(query, k=10)

    fused = reciprocal_rank_fusion([
        dense_rank,
        bm25_rank
    ])

    results = []
    seen = set()

    for idx, score in fused:

        if idx not in seen:

            results.append((chunks[idx], score))

            seen.add(idx)

        if len(results) >= k:
            break

    return results

In [ ]:
# RERANK RESULTS
def rerank_results(query, docs, top_n=5):

    pairs = [
    (query, doc.page_content)
    for doc, _ in docs
    ]

    scores = reranker.predict(pairs)

    ranked = sorted(
    zip(scores, [doc for doc, _ in docs]),
    key=lambda x: x[0],
    reverse=True
    )

    return ranked[:top_n]

In [ ]:
# SEARCH FUNCTION
def search_candidates(query):

    retrieved_docs = hybrid_search(
        query,
        k=10
    )

    reranked = rerank_results(
        query,
        retrieved_docs,
        top_n=5
    )

    context_parts = []

    for i, (score, doc) in enumerate(reranked):

        meta = doc.metadata

        context_parts.append(f"""
Candidate {i+1}

Name: {meta.get('name')}

Experience: {meta.get('years')} years

Source: {meta.get('source')}

Relevance Score: {round(float(score), 4)}

Resume:
{doc.page_content[:800]}
""")

    return "\n\n----------------\n\n".join(context_parts)

In [ ]:
query = """
Looking for a Java backend engineer with:

- Spring Boot
- AWS
- REST APIs
- Microservices
- SQL knowledge
"""

In [ ]:
# RETRIEVAL CONTEXT
retrieved_context = search_candidates(query)

In [ ]:
# for agent -> task -> create crew then process
from crewai import Agent, Task, Crew, Process

In [ ]:
# AGENT 1 — RETRIEVER
retriever_agent = Agent(

    role="Resume Retrieval Specialist",

    goal="Find resumes matching the job description",

    backstory="""
    You are an expert technical recruiter skilled at
    semantic resume search and candidate retrieval.
    """,

    llm=llm,

    verbose=True
)

In [ ]:
# AGENT 2 — TECHNICAL EVALUATOR
evaluator_agent = Agent(

    role="Technical Resume Evaluator",

    goal="Evaluate technical skills and strengths",

    backstory="""
    You are a senior backend engineer who evaluates
    Java, Spring Boot, AWS, SQL, and microservices expertise.
    """,

    llm=llm,

    verbose=True
)


In [ ]:
# AGENT 3 — RANKING AGENT
ranking_agent = Agent(

    role="Candidate Ranking Specialist",

    goal="Rank candidates from best to worst",

    backstory="""
    You compare candidates objectively based on
    technical fit and experience relevance.
    """,

    llm=llm,

    verbose=True
)

In [ ]:
# AGENT 4 — HR REVIEWER
review_agent = Agent(

    role="HR Hiring Reviewer",

    goal="Provide final hiring recommendation",

    backstory="""
    You finalize hiring decisions and provide
    concise HR recommendations.
    """,

    llm=llm,

    verbose=True
)


In [ ]:
# TASK 1 — RETRIEVE
retrieve_task = Task(

    description=f"""
    Analyze the following retrieved resume data:

    {retrieved_context}

    Find best matching candidates for this role:

    {query}
    """,

    expected_output="""
    Structured summary of top matching candidates.
    """,

    agent=retriever_agent
)

In [ ]:
# TASK 2 — TECHNICAL EVALUATION
evaluate_task = Task(

    description="""
    Evaluate each candidate technically.

    Check:
    - Java
    - Spring Boot
    - AWS
    - REST APIs
    - Microservices
    - SQL

    Mention strengths and weaknesses.
    """,

    expected_output="""
    Technical evaluation report for all candidates.
    """,

    context=[retrieve_task],

    agent=evaluator_agent
)

In [ ]:
# TASK 3 — RANKING
ranking_task = Task(

    description="""
    Rank all candidates from best to worst.

    Explain ranking decisions clearly.
    """,

    expected_output="""
    Final ranked list of candidates.
    """,

    context=[evaluate_task],

    agent=ranking_agent
)

In [ ]:
# TASK 4 — FINAL REVIEW
review_task = Task(

    description="""
    Provide final hiring recommendation.

    Mention:
    - Best candidate
    - Why selected
    - Hiring risks
    - Final shortlist
    """,

    expected_output="""
    Final hiring report.
    """,

    context=[ranking_task],

    agent=review_agent
)

In [ ]:
# CREATE CREW
crew = Crew(

    agents=[
        retriever_agent,
        evaluator_agent,
        ranking_agent,
        review_agent
    ],

    tasks=[
        retrieve_task,
        evaluate_task,
        ranking_task,
        review_task
    ],

    process=Process.sequential,

    verbose=True,

    memory=True
)

In [ ]:
# RUN CREW
result = crew.kickoff()

In [ ]:
print(result)